In [1]:
#!pip install sympy

In [2]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from qhdopt import QHD

In [3]:
# 初始化（用默认 3-bus 数据）
#model = SympyACOPFModel()   # 打开 proximal 选项，后面会用到

# 2bus model
Sbase_2 = 10.0
buses_2 = {
    1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
    2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
}
lines_2 = {
    1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase_2],
}
gens_2 = {
    1: [1, 0.0 / Sbase_2, 20.0 / Sbase_2, -20.0 / Sbase_2, 100.0 / Sbase_2, 0.00375, 2.0, 0.0],
}

model = SympyACOPFModel(Sbase = Sbase_2, buses=buses_2, lines=lines_2, gens=gens_2)

In [8]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()

rho = 10.0
alpha = 5e-1   # 对偶步长尽量小一点
max_outer = 1
tol = 1e-4

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=1500.0
        )

    option = 1
    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)

        qhd_model.simbi_setup(
            resolution=6,
            agents=1024,
            max_steps=1000,
            embedding_scheme="unary",
            post_processing_method='TNC',
            verbose=True
        )

        response = qhd_model.optimize(verbose=0)

        x_new = np.asarray(response.refined_minimizer)

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    PrintQHDACOPFResults(model, x_new)

    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1e4
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (7) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (6) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")


===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.5628805160522461
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0082	-0.0014	0.2934	0.0977	1.0000	0.0000	0.0000	0.0000
2	1.0064	-0.0100	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.2934	0.0977		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.0054	-0.0462	0.0203	0.0017


Total Ploss: -0.0408
Total Qloss: 0.0220
Total Load Supplied: 111.4164%
||h(x)|| = 0.38535346823990796
[rho-check] ||h_old||=4.939e-01, ||h_new||=3.854e-01, rho=10
Increasing rho to 20.0
Constraint check: False

===== End Loop =====



In [18]:
model.G_mat

array([[ 1.24373729, -1.24373729],
       [-1.24373729,  1.24373729]])

In [9]:
response.refined_minimizer


array([ 0.29344148,  0.09770903,  1.00821986,  1.00637924, -0.00140895,
       -0.00996584,  0.99457388,  1.00760601,  0.00536775,  0.02028874,
        0.        ])

In [16]:
qhd_model.func

0.001875*P_G0**2 + 2.0*P_G0 + 750.0*P_ij0**2 + 750.0*Q_ij0**2 + 755.0*S_tot_sq0**2 + 755.0*V_I0**2 + 750.0*V_I1**2 + 750.0*(P_G0 - 0.3)**2 + 750.0*(Q_G0 - 0.1)**2 + 750.0*(V_R0 - 1.0)**2 + 750.0*(V_R1 - 1.01)**2 + 750.0*(V_sq0 - 1.0)**2 + 750.0*(V_sq1 - 1.0201)**2 + 5.0*(-2.0*V_R0 + 1.0*V_sq0 + 1.0)**2 + 5.0*(-2.02*V_R1 + 1.0*V_sq1 + 1.0201)**2 + 5.0*(-1.25617466033865*V_I0 + 1.24373728746401*V_I1 + 5.14698113041411*V_R0 - 5.17733733962613*V_R1 - 0.0589352086958588)**2 + 5.0*(5.14698113041411*V_I0 - 5.09602092120209*V_I1 + 1.25617466033865*V_R0 - 1.26861203321329*V_R1 - 0.287438253396614)**2 + 5.0*(1.0*P_G0 - 5.14698113041411*V_I0 + 5.09602092120209*V_I1 - 1.23129991458937*V_R0 + 1.24373728746401*V_R1 - 0.0124373728746401)**2 + 5.0*(1.0*P_ij0 - 5.14698113041411*V_I0 + 5.09602092120209*V_I1 - 1.23129991458937*V_R0 + 1.24373728746401*V_R1 - 0.0124373728746401)**2 + 5.0*(1.0*Q_G0 + 1.25617466033865*V_I0 - 1.24373728746401*V_I1 - 5.02466071199007*V_R0 + 5.09602092120209*V_R1 - 0.0611602092

In [7]:
print(model.P_D, model.Q_D, model.P_G, model.Q_G)
print(model.variable_list)
print(model.Var_bound_list)

[0.  0.3] [0.  0.1] (P_G0,) (Q_G0,)
[P_G0, Q_G0, V_R0, V_R1, V_I0, V_I1, V_sq0, V_sq1, P_ij0, Q_ij0, S_tot_sq0]
[[0.0, 2.0], [-2.0, 10.0], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [0.81, 1.2100000000000002], [0.81, 1.2100000000000002], [-3.0, 3.0], [-3.0, 3.0], [0.0, 9.0]]
